# Lab 7 : Watch RAG fail on real-world edge cases

*W3 RAG Part 1 · Utrains LLMOps 8-Week Course*

Run each cell in order. Read the output. Move to the next.

See the matching slide in this week's concepts deck for the real-world story this lab teaches.

## What we are achieving in this lab

**Objective.** Point the Lab 6 loop at five failure modes that show up in production. This lab is the reason Week 4 exists.

**Prerequisites.** Lab 6 finished. You can draw load → split → embed → store → retrieve → generate.

**What you will do.**

1. Exact token (`E4221`): embeddings may bury the runbook; keywords would not.
2. Duplicates: identical chunks steal two slots in top-k.
3. Version conflict: two refund policies, no metadata to pick the current one.
4. Off-topic neighbour: a historical outage looks relevant to "why did payments fail?"
5. Missing fact: the retriever still returns chunks; a naive generator will invent the rest.

**What you should see.**

Not a perfect demo. A list of bugs with names you can use in an interview. The fixes (hybrid search, reranking, metadata filters, better chunking, RAGAS-style evals) are Week 4. Today you only need to recognise the bugs.

### Setup. A hostile little corpus

Each document is a real shape of production junk: an error code, a duplicate, two policy versions, a stale incident, and a PTO blurb that will get retrieved when nothing else matches.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import ChatOllama, OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")
llm = ChatOllama(model="llama3.2:1b", temperature=0)

CORPUS = [
    Document(page_content="Error E4221 occurs when the payment service receives a malformed VAT field.", metadata={"id": "e4221-explain"}),
    Document(page_content="Error E4221: see runbook RB-019.", metadata={"id": "e4221-runbook"}),
    Document(page_content="Refund policy: 30 days. Submit ticket within window.", metadata={"id": "refund-v1-a", "version": "1"}),
    Document(page_content="Refund policy: 30 days. Submit ticket within window.", metadata={"id": "refund-v1-b", "version": "1"}),
    Document(page_content="Refund policy v2 (in effect from January 2026): 14 days.", metadata={"id": "refund-v2", "version": "2"}),
    Document(page_content="Our annual outage in 2023 caused payment failures across the region.", metadata={"id": "incident-2023"}),
    Document(page_content="PTO is unlimited with a 2-week annual minimum.", metadata={"id": "pto"}),
]

vectorstore = InMemoryVectorStore.from_documents(CORPUS, embedding=embeddings)


def show_hits(title: str, query: str, k: int = 3) -> list[Document]:
    hits = vectorstore.as_retriever(search_kwargs={"k": k}).invoke(query)
    print(f"--- {title} ---")
    print(f"query: {query}")
    for i, doc in enumerate(hits, start=1):
        print(f"  {i}. [{doc.metadata.get('id')}]  {doc.page_content}")
    print()
    return hits


print(f"indexed {len(CORPUS)} documents")

### 7.A  Exact tokens: error codes, SKUs, names

People paste `E4221` into the search box. Keyword search is built for that. Dense embeddings sometimes still rank both E4221 docs well because the code is rare. Change the code to a buried SKU like `PROD-A412-X3` inside a long paragraph and dense search starts to miss.

Watch what comes back. Then imagine the runbook doc did not contain the words "error" or "VAT" — only the code and a filename.

In [ ]:
show_hits("7.A  exact-match query", "What does error code E4221 mean?")

**Production fix (Week 4):** hybrid search (BM25 + vectors + reciprocal rank fusion). BM25 catches the token. Dense search catches the paraphrase "VAT field is malformed." You want both.

### 7.B  Duplicates steal top-k slots

Two copies of the same 30-day refund sentence. They get (almost) the same score. They occupy two of your three slots. The v2 policy may get pushed down or out.

In [ ]:
show_hits("7.B  duplicates in top-k", "How long do I have to request a refund?", k=3)

**Production fix:** deduplicate *before* you embed (hash the text, or near-duplicate with a similarity cap). Rerankers can penalise clones, but not storing the clone is cheaper.

### 7.C  Version conflict

The 30-day policy and the 14-day policy are both "about refunds." Without `valid_from` / `version` metadata and a filter, the retriever cannot prefer January 2026. The generator will pick one, blend them, or hedge. All three are wrong in a regulated setting.

In [ ]:
show_hits("7.C  two policies, one question", "How long do I have to request a refund?", k=5)

**Production fix:** store version and dates in metadata. Filter at retrieval (`version = current`, `valid_from <= today`). Retrieval is not only cosine; it is cosine plus predicates.

### 7.D  Off-topic neighbour

"Why did payments fail?" is about a live incident in the user's head. The 2023 outage doc is about payments failing. It will get retrieved. A model that cannot tell *historical* from *current* will explain today's checkout bug with a three-year-old postmortem.

In [ ]:
show_hits("7.D  stale incident looks relevant", "Why did payments fail?")

**Production fix:** recency filters, source-type metadata (`runbook` vs `incident-review`), cross-encoder reranking that sees the query and the document together instead of only cosine against an independent vector.

### 7.E  Missing fact + a generator that hates silence

There is no parental-leave document in this corpus. The retriever still returns its best three guesses. PTO is a cousin of leave, so it often wins. Hand those chunks to a model with a weak prompt and you get an invented policy.

This is the shape of *Mata v. Avianca*: fluent, specific, false.

In [ ]:
hits = show_hits("7.E  question not in the corpus", "What is the company's parental leave policy?")

context = "\n\n---\n\n".join(doc.page_content for doc in hits)
naive = llm.invoke(
    [
        ("system", "You are a helpful HR assistant. Answer the question."),
        ("human", f"Context:\n{context}\n\nQuestion: What is the company's parental leave policy?"),
    ]
)
grounded = llm.invoke(
    [
        (
            "system",
            "You answer using ONLY the provided context. "
            "If the context does not contain the answer, say you cannot find it. Do not guess.",
        ),
        ("human", f"Context:\n{context}\n\nQuestion: What is the company's parental leave policy?"),
    ]
)

print("--- naive prompt (no escape hatch) ---")
print(naive.content)
print("\n--- grounded prompt ---")
print(grounded.content)

`llama3.2:1b` may still guess after the grounded prompt. That is data. A prompt is not a guarantee. Production adds a **numeric** gate: if the top score is below a threshold you measured, you do not call the generator; you return "I don't know" from *your* code.

The handbook in Lab 6 *does* contain parental leave. This corpus does not. Same question, different index, different truth. That is why you evaluate on *your* documents.

## Five fixes that land in Week 4

| Failure you just saw | What Week 4 adds |
|----------------------|------------------|
| Exact token miss | Hybrid search: BM25 + vectors + RRF |
| Duplicate chunks | Dedup at index time; rerank |
| Two versions of a policy | Metadata filters (version, date, tenant) |
| Stale / off-topic neighbour | Reranker + recency + source type |
| Confident answer to a missing fact | Similarity threshold, citations, RAGAS-style evals |

If someone asks in an interview whether you "know RAG," do not describe Lab 6's happy path. Describe Lab 7, then say what you would measure next.

You can now answer Week 3's question: **yes, you can make a model speak about your data** — and you know the ways that sentence is incomplete.